## New script

In [2]:
!pip install rosbags

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.8/144.8 KB 1.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 KB 2.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 9.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 10.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.2 MB/s eta 0:00:00a 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
from pathlib import Path
from rosbags.highlevel import AnyReader

BAG_ROOT = Path("/home/soehn/work/tsl")
bags = sorted(BAG_ROOT.rglob("*.bag"))

bag = bags[0]
print(bag)

with AnyReader([bag]) as reader:
    for c in reader.connections:
        print(c.topic, c.msgtype, c.msgcount)

/home/soehn/work/tsl/1_worldcooking/1-m.bag
/file_version std_msgs/msg/UInt32 1
/device_0/sensor_0/Color_0/info realsense_msgs/msg/StreamInfo 1
/device_0/sensor_0/Color_0/info/camera_info sensor_msgs/msg/CameraInfo 1
/device_0/info diagnostic_msgs/msg/KeyValue 6
/device_0/sensor_0/info diagnostic_msgs/msg/KeyValue 1
/device_0/sensor_0/option/Brightness/value std_msgs/msg/Float32 1
/device_0/sensor_0/option/Brightness/description std_msgs/msg/String 1
/device_0/sensor_0/option/Contrast/value std_msgs/msg/Float32 1
/device_0/sensor_0/option/Contrast/description std_msgs/msg/String 1
/device_0/sensor_0/option/Hue/value std_msgs/msg/Float32 1
/device_0/sensor_0/option/Hue/description std_msgs/msg/String 1
/device_0/sensor_0/option/Saturation/value std_msgs/msg/Float32 1
/device_0/sensor_0/option/Saturation/description std_msgs/msg/String 1
/device_0/sensor_0/option/Frames_Queue_Size/value std_msgs/msg/Float32 1
/device_0/sensor_0/option/Frames_Queue_Size/description std_msgs/msg/String 1
/

In [4]:
with AnyReader([bag]) as reader:
    image_connections = [
        c for c in reader.connections
        if "Image" in c.msgtype or "image" in c.topic.lower()
    ]

    for c in image_connections:
        print(c.topic, c.msgtype, c.msgcount)

/device_0/sensor_0/Color_0/image/data sensor_msgs/msg/Image 340
/device_0/sensor_0/Color_0/image/metadata diagnostic_msgs/msg/KeyValue 1700


In [5]:
import numpy as np
import pandas as pd

COLOR_TOPIC = "/device_0/sensor_1/Color_0/image/data"  # change if your topic is different

def inspect_bag_fps_rosbags(bag_path, topic=COLOR_TOPIC):
    bag_path = Path(bag_path)
    timestamps_ns = []

    with AnyReader([bag_path]) as reader:
        conns = [c for c in reader.connections if c.topic == topic]

        if not conns:
            return {
                "bag": str(bag_path),
                "status": f"topic_not_found: {topic}",
            }

        for conn, timestamp_ns, rawdata in reader.messages(connections=conns):
            timestamps_ns.append(timestamp_ns)

    if len(timestamps_ns) < 2:
        return {
            "bag": str(bag_path),
            "frame_count": len(timestamps_ns),
            "status": "too_few_frames",
        }

    ts = np.array(timestamps_ns, dtype=np.float64) / 1e9
    gaps = np.diff(ts)

    return {
        "bag": str(bag_path),
        "relative_bag": str(bag_path.relative_to(BAG_ROOT)),
        "frame_count": len(ts),
        "duration_sec": round(ts[-1] - ts[0], 4),
        "avg_fps": round((len(ts) - 1) / (ts[-1] - ts[0]), 3),
        "median_fps": round(1.0 / np.median(gaps), 3),
        "min_gap_ms": round(float(np.min(gaps) * 1000), 3),
        "median_gap_ms": round(float(np.median(gaps) * 1000), 3),
        "max_gap_ms": round(float(np.max(gaps) * 1000), 3),
        "status": "ok",
    }

inspect_bag_fps_rosbags(bags[0])

{'bag': '/home/soehn/work/tsl/1_worldcooking/1-m.bag',
 'status': 'topic_not_found: /device_0/sensor_1/Color_0/image/data'}

In [6]:
COLOR_TOPIC = "/device_0/sensor_0/Color_0/image/data"

inspect_bag_fps_rosbags(bags[0], topic=COLOR_TOPIC)

{'bag': '/home/soehn/work/tsl/1_worldcooking/1-m.bag',
 'relative_bag': '1_worldcooking/1-m.bag',
 'frame_count': 340,
 'duration_sec': np.float64(10.0469),
 'avg_fps': np.float64(33.742),
 'median_fps': np.float64(30.758),
 'min_gap_ms': 16.812,
 'median_gap_ms': 32.512,
 'max_gap_ms': 41.084,
 'status': 'ok'}

In [8]:
import struct
import numpy as np
import cv2
from pathlib import Path
from rosbags.highlevel import AnyReader

COLOR_TOPIC = "/device_0/sensor_0/Color_0/image/data"


def read_u32(buf, pos):
    return struct.unpack_from("<I", buf, pos)[0], pos + 4


def read_u8(buf, pos):
    return struct.unpack_from("<B", buf, pos)[0], pos + 1


def read_string(buf, pos):
    n, pos = read_u32(buf, pos)
    s = buf[pos:pos+n].decode("utf-8")
    return s, pos + n


def parse_ros1_image(rawdata):
    buf = bytes(rawdata)
    pos = 0

    # std_msgs/Header
    seq, pos = read_u32(buf, pos)
    stamp_sec, pos = read_u32(buf, pos)
    stamp_nsec, pos = read_u32(buf, pos)
    frame_id, pos = read_string(buf, pos)

    height, pos = read_u32(buf, pos)
    width, pos = read_u32(buf, pos)
    encoding, pos = read_string(buf, pos)
    is_bigendian, pos = read_u8(buf, pos)
    step, pos = read_u32(buf, pos)

    data_len, pos = read_u32(buf, pos)
    data = buf[pos:pos + data_len]

    return {
        "seq": seq,
        "stamp_sec": stamp_sec,
        "stamp_nsec": stamp_nsec,
        "frame_id": frame_id,
        "height": height,
        "width": width,
        "encoding": encoding,
        "is_bigendian": is_bigendian,
        "step": step,
        "data": data,
        "data_len": data_len,
        "raw_len": len(buf),
    }

In [9]:
with AnyReader([bags[0]]) as reader:
    conns = [c for c in reader.connections if c.topic == COLOR_TOPIC]

    for conn, timestamp_ns, rawdata in reader.messages(connections=conns):
        img_msg = parse_ros1_image(rawdata)
        print("height:", img_msg["height"])
        print("width:", img_msg["width"])
        print("encoding:", img_msg["encoding"])
        print("step:", img_msg["step"])
        print("data length:", img_msg["data_len"])
        print("raw length:", img_msg["raw_len"])
        break

height: 1080
width: 1920
encoding: rgb8
step: 5760
data length: 6220800
raw length: 6220846


In [10]:
def image_msg_to_bgr(img_msg):
    h = img_msg["height"]
    w = img_msg["width"]
    encoding = img_msg["encoding"].lower()
    data = img_msg["data"]

    if encoding == "rgb8":
        img = np.frombuffer(data, dtype=np.uint8).reshape(h, w, 3)
        return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    if encoding == "bgr8":
        return np.frombuffer(data, dtype=np.uint8).reshape(h, w, 3)

    if encoding == "mono8":
        img = np.frombuffer(data, dtype=np.uint8).reshape(h, w)
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    if encoding in ["rgba8", "bgra8"]:
        img = np.frombuffer(data, dtype=np.uint8).reshape(h, w, 4)
        if encoding == "rgba8":
            return cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
        return cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

    raise ValueError(f"Unsupported encoding: {img_msg['encoding']}")

In [11]:
REPORT_ROOT = Path("/home/soehn/dataset_arch/SignLanguage")
MP4_ROOT = REPORT_ROOT / "mp4_from_bag"
MP4_ROOT.mkdir(parents=True, exist_ok=True)


def bag_to_mp4_rosbags(bag_path, fps=None, topic=COLOR_TOPIC):
    bag_path = Path(bag_path)

    if fps is None:
        info = inspect_bag_fps_rosbags(bag_path, topic=topic)
        fps = float(info["median_fps"] or info["avg_fps"] or 30.0)

    out_path = MP4_ROOT / bag_path.relative_to(BAG_ROOT).with_suffix(".mp4")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    writer = None
    frames_written = 0

    with AnyReader([bag_path]) as reader:
        conns = [c for c in reader.connections if c.topic == topic]

        for conn, timestamp_ns, rawdata in reader.messages(connections=conns):
            img_msg = parse_ros1_image(rawdata)
            frame = image_msg_to_bgr(img_msg)

            h, w = frame.shape[:2]

            if writer is None:
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))

            writer.write(frame)
            frames_written += 1

    if writer is not None:
        writer.release()

    return {
        "bag": str(bag_path),
        "mp4": str(out_path),
        "fps_used": fps,
        "frames_written": frames_written,
    }

In [12]:
bag_to_mp4_rosbags(bags[0])

{'bag': '/home/soehn/work/tsl/1_worldcooking/1-m.bag',
 'mp4': '/home/soehn/dataset_arch/SignLanguage/mp4_from_bag/1_worldcooking/1-m.mp4',
 'fps_used': 30.758,
 'frames_written': 340}

In [13]:
cap = cv2.VideoCapture(str(MP4_ROOT / bags[0].relative_to(BAG_ROOT).with_suffix(".mp4")))
print("opened:", cap.isOpened())
print("fps:", cap.get(cv2.CAP_PROP_FPS))
print("frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("width:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("height:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

opened: True
fps: 30.758
frames: 340.0
width: 1920.0
height: 1080.0


## One folder test

In [14]:
ONE_FOLDER = Path("/home/soehn/work/tsl/1_worldcooking")
one_folder_bags = sorted(ONE_FOLDER.glob("*.bag"))

print("Bag files:", len(one_folder_bags))
print(one_folder_bags[:5])

Bag files: 253
[PosixPath('/home/soehn/work/tsl/1_worldcooking/1-m.bag'), PosixPath('/home/soehn/work/tsl/1_worldcooking/10-m.bag'), PosixPath('/home/soehn/work/tsl/1_worldcooking/100-m.bag'), PosixPath('/home/soehn/work/tsl/1_worldcooking/101-m.bag'), PosixPath('/home/soehn/work/tsl/1_worldcooking/102-m.bag')]


In [15]:
rows = []

for i, bag in enumerate(one_folder_bags):
    print(f"{i + 1}/{len(one_folder_bags)} {bag.name}")
    try:
        rows.append(inspect_bag_fps_rosbags(bag, topic=COLOR_TOPIC))
    except Exception as e:
        rows.append({
            "bag": str(bag),
            "relative_bag": str(bag.relative_to(BAG_ROOT)),
            "status": f"error: {type(e).__name__}: {e}",
        })

fps_1_worldcooking_df = pd.DataFrame(rows)
fps_1_worldcooking_df

1/253 1-m.bag
2/253 10-m.bag
3/253 100-m.bag
4/253 101-m.bag
5/253 102-m.bag
6/253 103-m.bag
7/253 104-m.bag
8/253 105-m.bag
9/253 106-m.bag
10/253 107-m.bag
11/253 108-m.bag
12/253 109-m.bag
13/253 11-m.bag
14/253 110-m.bag
15/253 111-m.bag
16/253 112-m.bag
17/253 113-m.bag
18/253 114-m.bag
19/253 115-m.bag
20/253 116-m.bag
21/253 117-m.bag
22/253 118-m.bag
23/253 119-m.bag
24/253 12-m.bag
25/253 120-m.bag
26/253 121-m.bag
27/253 122-m.bag
28/253 123-m.bag
29/253 124-m.bag
30/253 125-m.bag
31/253 126-m.bag
32/253 127-m.bag
33/253 128-m.bag
34/253 129-m.bag
35/253 13-m.bag
36/253 130-m.bag
37/253 131-m.bag
38/253 132-m.bag
39/253 133-m.bag
40/253 134-m.bag
41/253 135-m.bag
42/253 136-m.bag
43/253 137-m.bag
44/253 138-m.bag
45/253 139-m.bag
46/253 14-m.bag
47/253 140-m.bag
48/253 141-m.bag
49/253 142-m.bag
50/253 143-m.bag
51/253 144-m.bag
52/253 145-m.bag
53/253 146-m.bag
54/253 147-m.bag
55/253 148-m.bag
56/253 149-m.bag
57/253 15-m.bag
58/253 150-m.bag
59/253 151-m.bag
60/253 152-m.b

,bag,relative_bag,frame_count,duration_sec,avg_fps,median_fps,min_gap_ms,median_gap_ms,max_gap_ms,status
0,/home/soehn/work/tsl/1_worldcooking/1-m.bag,1_worldcooking/1-m.bag,340,10.0469,33.742,30.758,16.812,32.512,41.084,ok
1,/home/soehn/work/tsl/1_worldcooking/10-m.bag,1_worldcooking/10-m.bag,383,11.0809,34.474,33.821,18.069,29.567,41.644,ok
2,/home/soehn/work/tsl/1_worldcooking/100-m.bag,1_worldcooking/100-m.bag,442,11.4710,38.445,37.902,18.244,26.384,39.953,ok
3,/home/soehn/work/tsl/1_worldcooking/101-m.bag,1_worldcooking/101-m.bag,548,14.4024,37.980,37.428,18.107,26.718,36.619,ok
4,/home/soehn/work/tsl/1_worldcooking/102-m.bag,1_worldcooking/102-m.bag,446,11.6255,38.278,37.575,18.367,26.614,36.524,ok
...,...,...,...,...,...,...,...,...,...,...
248,/home/soehn/work/tsl/1_worldcooking/95-m.bag,1_worldcooking/95-m.bag,525,13.7248,38.179,37.904,17.912,26.382,41.222,ok
249,/home/soehn/work/tsl/1_worldcooking/96-m.bag,1_worldcooking/96-m.bag,348,10.3084,33.662,34.038,18.724,29.378,42.487,ok
250,/home/soehn/work/tsl/1_worldcooking/97-m.bag,1_worldcooking/97-m.bag,503,13.0682,38.414,38.118,18.865,26.235,43.648,ok
251,/home/soehn/work/tsl/1_worldcooking/98-m.bag,1_worldcooking/98-m.bag,557,14.5102,38.318,37.983,17.846,26.327,37.536,ok


In [16]:
folder_report_dir = REPORT_ROOT / "reports" / "1_worldcooking"
folder_report_dir.mkdir(parents=True, exist_ok=True)

fps_report_path = folder_report_dir / "bag_real_fps_report.csv"
fps_1_worldcooking_df.to_csv(fps_report_path, index=False)

print(fps_report_path)
fps_1_worldcooking_df.describe()

/home/soehn/dataset_arch/SignLanguage/reports/1_worldcooking/bag_real_fps_report.csv


,frame_count,duration_sec,avg_fps,median_fps,min_gap_ms,median_gap_ms,max_gap_ms
count,253.000000,253.000000,253.000000,253.000000,253.000000,253.000000,253.000000
mean,406.201581,10.448392,38.919075,38.272087,18.546870,26.265186,49.534415
std,77.000534,2.068163,2.336370,2.577985,0.661013,2.042796,142.915804
min,246.000000,5.949500,29.031000,27.749000,16.555000,24.078000,30.240000
25%,352.000000,8.985500,38.287000,37.820000,18.142000,25.125000,35.072000
50%,395.000000,10.183200,39.581000,39.082000,18.726000,25.587000,37.203000
75%,450.000000,11.625500,40.485000,39.801000,19.007000,26.441000,40.215000
max,701.000000,18.624300,42.195000,41.531000,20.370000,36.038000,2158.457000


In [17]:
convert_rows = []

for i, bag in enumerate(one_folder_bags):
    print(f"{i + 1}/{len(one_folder_bags)} {bag.name}")
    try:
        convert_rows.append(bag_to_mp4_rosbags(bag, topic=COLOR_TOPIC))
    except Exception as e:
        convert_rows.append({
            "bag": str(bag),
            "mp4": None,
            "fps_used": None,
            "frames_written": None,
            "status": f"error: {type(e).__name__}: {e}",
        })

convert_1_worldcooking_df = pd.DataFrame(convert_rows)
convert_1_worldcooking_df

1/253 1-m.bag
2/253 10-m.bag
3/253 100-m.bag
4/253 101-m.bag
5/253 102-m.bag
6/253 103-m.bag
7/253 104-m.bag
8/253 105-m.bag
9/253 106-m.bag
10/253 107-m.bag
11/253 108-m.bag
12/253 109-m.bag
13/253 11-m.bag
14/253 110-m.bag
15/253 111-m.bag
16/253 112-m.bag
17/253 113-m.bag
18/253 114-m.bag
19/253 115-m.bag
20/253 116-m.bag
21/253 117-m.bag
22/253 118-m.bag
23/253 119-m.bag
24/253 12-m.bag
25/253 120-m.bag
26/253 121-m.bag
27/253 122-m.bag
28/253 123-m.bag
29/253 124-m.bag
30/253 125-m.bag
31/253 126-m.bag
32/253 127-m.bag
33/253 128-m.bag
34/253 129-m.bag
35/253 13-m.bag
36/253 130-m.bag
37/253 131-m.bag
38/253 132-m.bag
39/253 133-m.bag
40/253 134-m.bag
41/253 135-m.bag
42/253 136-m.bag
43/253 137-m.bag
44/253 138-m.bag
45/253 139-m.bag
46/253 14-m.bag
47/253 140-m.bag
48/253 141-m.bag
49/253 142-m.bag
50/253 143-m.bag
51/253 144-m.bag
52/253 145-m.bag
53/253 146-m.bag
54/253 147-m.bag
55/253 148-m.bag
56/253 149-m.bag
57/253 15-m.bag
58/253 150-m.bag
59/253 151-m.bag
60/253 152-m.b

,bag,mp4,fps_used,frames_written
0,/home/soehn/work/tsl/1_worldcooking/1-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,30.758,340
1,/home/soehn/work/tsl/1_worldcooking/10-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,33.821,383
2,/home/soehn/work/tsl/1_worldcooking/100-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,37.902,442
3,/home/soehn/work/tsl/1_worldcooking/101-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,37.428,548
4,/home/soehn/work/tsl/1_worldcooking/102-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,37.575,446
...,...,...,...,...
248,/home/soehn/work/tsl/1_worldcooking/95-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,37.904,525
249,/home/soehn/work/tsl/1_worldcooking/96-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,34.038,348
250,/home/soehn/work/tsl/1_worldcooking/97-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,38.118,503
251,/home/soehn/work/tsl/1_worldcooking/98-m.bag,/home/soehn/dataset_arch/SignLanguage/mp4_from...,37.983,557


In [18]:
convert_report_path = folder_report_dir / "bag_to_mp4_report.csv"
convert_1_worldcooking_df.to_csv(convert_report_path, index=False)

print(convert_report_path)

/home/soehn/dataset_arch/SignLanguage/reports/1_worldcooking/bag_to_mp4_report.csv


In [19]:
converted_mp4s = sorted((MP4_ROOT / "1_worldcooking").glob("*.mp4"))
print("MP4 files:", len(converted_mp4s))

for p in converted_mp4s[:5]:
    cap = cv2.VideoCapture(str(p))
    print(
        p.name,
        "opened=", cap.isOpened(),
        "fps=", cap.get(cv2.CAP_PROP_FPS),
        "frames=", cap.get(cv2.CAP_PROP_FRAME_COUNT),
        "size=", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    )
    cap.release()

MP4 files: 253
1-m.mp4 opened= True fps= 30.758 frames= 340.0 size= 1920 x 1080
10-m.mp4 opened= True fps= 33.82 frames= 383.0 size= 1920 x 1080
100-m.mp4 opened= True fps= 37.902 frames= 442.0 size= 1920 x 1080
101-m.mp4 opened= True fps= 37.428 frames= 548.0 size= 1920 x 1080
102-m.mp4 opened= True fps= 37.575 frames= 446.0 size= 1920 x 1080
